In [ ]:
"""
breakup_manual_review.py
────────────────────────
Interactive OpenCV tool for manually reviewing and correcting break-up dates.

CONTROLS
  ← / A       Move break-up divider left  (one image)
  → / D       Move break-up divider right (one image)
  ]            Toggle "show all" overview (all images at once for scanning)
  Enter        Confirm position and save result, advance to next lake-year
  X            Mark as NO MANUAL DATE (no detectable break-up), advance
  B            Go back one lake-year (undo last save)
  O            Toggle lake outline overlay
  Z            Toggle tight zoom (40 m buffer vs 500 m default)
  ESC          Quit (progress already saved up to this point)

Only 8 images are shown at a time so each thumbnail stays large and readable.
Moving the divider past the visible window scrolls the strip by 4 images,
revealing the next batch -- this lets a long time series be narrowed down
quickly. Press ] to expand to the full time series for orientation.

The divider can be placed BEFORE the first image (break-up already occurred
before the observation window) or AFTER the last image (break-up hasn't
happened yet in the observation window).

Observations whose lake polygon falls entirely under the TIF's nodata border
(no valid pixels beneath the lake itself) are dropped from the strip rather
than shown as black/empty cells.

Valid observations are read directly from the ObsK columns in the break-up
results CSV (Obs1, Obs2, ... ObsN with paired ObsKDate / ObsKPctIce /
ObsKPctWat columns).  Each stem is still checked against the cloud
classifications; any cloudy images that were skipped are shown in the header
so the reviewer is aware.

OUTPUT CSV columns
  study_site | lake_id | year | original_breakup_date |
  manual_breakup_date | image_before | image_after
"""

import os
import glob
import queue
import threading
import warnings
import csv

import numpy as np
import pandas as pd
import cv2
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
#  USER CONFIG  --  paths to local data on this machine
# ══════════════════════════════════════════════════════════════════════════════

_BREAKUP_OUT    = r"E:\planetscope_lake_ice\Data\Output\Breakup"
_BREAKUP_DATES  = r"C:\Users\nj142\Desktop\Time Series\PS\Breakup"

SITES = ["NS", "YF", "YKD"]

BREAKUP_RESULTS_CSVS = {
    site: os.path.join(_BREAKUP_DATES, f"{site}_breakup_dates.csv")
    for site in SITES
}

_INPUT_BASE  = r"E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data"


CLOUD_CSVS = [
    os.path.join(_INPUT_BASE, f"{site}_cloud_classifications.csv") for site in SITES
]

SITE_FOLDERS = {
    site: os.path.join(_INPUT_BASE, f"{site} 50x50 km") for site in SITES
}

SITE_SHAPEFILES = {
    site: os.path.join(
        _INPUT_BASE, f"{site} 50x50 km",
        f"{site} Lakes from ALPOD Shapefile", f"{site}_50x50km_lakes.shp"
    ) for site in SITES
}

OUTPUT_CSV = os.path.join(_BREAKUP_OUT, "manual_breakup_review.csv")

# ── Display ──────────────────────────────────────────────────────────────────
MAX_WINDOW_W   = 1800
MAX_WINDOW_H   = 900
DIVIDER_W      = 44
LABEL_H        = 62
HEADER_H       = 52
MIN_THUMB_W    = 90

# Sliding-window display. 8 images visible at a time; when the divider tries
# to leave the visible strip the window shifts by 4 to reveal the next batch.
WINDOW_SIZE    = 8
WINDOW_SHIFT   = 4

WATER_THRESHOLD    = 75.0
BOUNDARY_BUFFER_M  = 500
ZOOM_TIGHT_BUFFER  = 40
PRELOAD_AHEAD      = 10

# ── Colours (BGR) ────────────────────────────────────────────────────────────
C_BG          = (26,  13,  13)
C_CELL        = (38,  22,  22)
C_HEADER      = (42,  20,  10)
C_DIVIDER     = (0,  140, 220)
C_WATER_HIGH  = (255, 170, 60)
C_WATER_LOW   = (180, 200, 220)
C_WHITE       = (255, 255, 255)
C_DIM         = (160, 160, 190)
C_CONFIRMED   = (60, 200, 80)
C_WARN        = (50, 180, 255)   # amber for "cloudy skipped" notice


# ══════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _norm_id(v):
    s = str(v)
    return s[:-2] if s.endswith(".0") else s


def _strip_tif_suffix(fname):
    """
    Strip everything from '_ortho_analytic' onward so that full TIF filenames
    and the bare stems stored in ObsK columns resolve to the same key.
    e.g. '20190420_205400_1035_ortho_analytic_4b_sr.tif' -> '20190420_205400_1035'
    """
    return fname.rsplit("_ortho_analytic", 1)[0]


def _stretch(band, lo=2, hi=98):
    band  = band.astype(np.float32)
    valid = band > 0
    if not valid.any():
        return np.zeros_like(band, dtype=np.uint8)
    p_lo, p_hi = np.percentile(band[valid], (lo, hi))
    if p_hi <= p_lo:
        return np.zeros_like(band, dtype=np.uint8)
    out = np.clip((band - p_lo) / (p_hi - p_lo) * 255.0, 0, 255)
    out[~valid] = 0
    return out.astype(np.uint8)


def _put_text(img, text, x, y, scale=0.45, colour=C_WHITE, thickness=1):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                scale, colour, thickness, cv2.LINE_AA)


def letterbox_frame(frame, win_w, win_h):
    fh, fw = frame.shape[:2]
    if fw == 0 or fh == 0 or win_w == 0 or win_h == 0:
        return frame
    scale = min(win_w / fw, win_h / fh)
    new_w = int(fw * scale)
    new_h = int(fh * scale)
    interp = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(frame, (new_w, new_h), interpolation=interp)
    canvas = np.zeros((win_h, win_w, 3), dtype=np.uint8)
    x0 = (win_w - new_w) // 2
    y0 = (win_h - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_very_cloudy(cloud_csvs=CLOUD_CSVS):
    """
    Returns a set of image stems (everything before '_ortho_analytic').
    ObsK values (e.g. '20190420_205400_1035') are already in this format so
    membership testing is a plain `stem in very_cloudy` check.
    """
    names = set()
    for path in cloud_csvs:
        if not os.path.exists(path):
            print(f"  WARNING: cloud CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        mask = df["Classification"].str.strip() == "Very Cloudy"
        for fname in df.loc[mask, "Filename"].tolist():
            names.add(_strip_tif_suffix(str(fname)))
    print(f"  {len(names):,} Very Cloudy image stems loaded.")
    return names


def load_results(results_csvs):
    """
    Load break-up results CSVs.  ALL columns are preserved so that the ObsK /
    ObsKDate / ObsKPctIce / ObsKPctWat columns are available for obs parsing
    downstream in parse_obs_from_row().
    """
    frames = []
    for site, path in results_csvs.items():
        if not os.path.exists(path):
            print(f"  WARNING: results CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        df["study_site"]   = site
        df["lake_id"]      = df["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
        df["year"]         = pd.to_numeric(df["year"], errors="coerce")
        df["breakup_date"] = pd.to_datetime(df["breakup_date"], errors="coerce")
        if "obs_before_date" in df.columns:
            df["obs_before_date"] = pd.to_datetime(df["obs_before_date"], errors="coerce")
        df = df[df["year"].notna()].copy()
        df["year"] = df["year"].astype(int)
        frames.append(df)
        print(f"  {site}: {len(df):,} rows")
    if not frames:
        raise RuntimeError("No break-up results CSVs loaded.")
    out = pd.concat(frames, ignore_index=True)
    out = out[out["breakup_date"].notna()].reset_index(drop=True)
    print(f"  -> {len(out):,} valid break-up results.")
    return out


# ══════════════════════════════════════════════════════════════════════════════
#  OBS PARSING FROM ObsK COLUMNS
# ══════════════════════════════════════════════════════════════════════════════

def parse_obs_from_row(fu_row, very_cloudy):
    """
    Walk Obs1, Obs2, ... ObsN columns in a results-row dict and return
    (obs_list, cloudy_skipped).

    Each entry in obs_list is a dict:
        image_name     -- ObsK stem  (e.g. '20190420_205400_1035')
        datetime       -- pd.Timestamp parsed from ObsKDate
        water_fraction -- float from ObsKPctWat
        ice_fraction   -- float from ObsKPctIce
        tif_filename   -- same as image_name initially; resolved to the real
                          TIF basename in prepare_lake_year once img_lookup
                          is available

    Cloud check: a stem is considered cloudy when it matches the FULL leading
    portion of any Very Cloudy filename up to '_ortho_analytic'.  Because
    load_very_cloudy() already strips that suffix, `stem in very_cloudy` is
    an exact match -- '20190420_205400_1035' matches
    '20190420_205400_1035_ortho_analytic_4b_sr.tif' but NOT
    '20190420_205400_1043_ortho_analytic_4b_sr.tif'.
    """
    obs_list       = []
    cloudy_skipped = 0
    k = 1
    while True:
        stem_col = f"Obs{k}"
        date_col = f"Obs{k}Date"
        wat_col  = f"Obs{k}PctWat"
        ice_col  = f"Obs{k}PctIce"

        # Stop when the column no longer exists in this row
        if stem_col not in fu_row:
            break

        stem = fu_row.get(stem_col)
        if pd.isna(stem) or str(stem).strip() == "":
            break       # trailing blank columns -- end of obs list for this year
        stem = str(stem).strip()

        if stem in very_cloudy:
            cloudy_skipped += 1
            k += 1
            continue

        # Parse date
        date_val = fu_row.get(date_col)
        try:
            dt = pd.Timestamp(date_val)
        except Exception:
            dt = pd.NaT

        # Parse percentages safely
        def _safe_float(v):
            try:
                return float(v)
            except (TypeError, ValueError):
                return 0.0

        wat = _safe_float(fu_row.get(wat_col, 0.0))
        ice = _safe_float(fu_row.get(ice_col, 0.0))

        obs_list.append({
            "image_name":     stem,
            "datetime":       dt,
            "water_fraction": wat,
            "ice_fraction":   ice,
            "tif_filename":   stem,   # placeholder; resolved below
        })
        k += 1

    return obs_list, cloudy_skipped


# ══════════════════════════════════════════════════════════════════════════════
#  IMAGE LOOKUP
# ══════════════════════════════════════════════════════════════════════════════

def build_image_lookup(site_folder):
    """
    Recursively index every .tif under site_folder.
    Two keys per file: full basename AND the stripped stem so that ObsK
    stems resolve correctly via img_lookup.get(stem).
    """
    lookup = {}
    for p in glob.glob(os.path.join(site_folder, "**", "*.tif"), recursive=True):
        fname = os.path.basename(p)
        lookup.setdefault(fname, p)
        lookup.setdefault(_strip_tif_suffix(fname), p)
    return lookup


def build_lake_year_list(df_results, sample_per_site=250, seed=42):
    rows = df_results.sort_values(["study_site", "lake_id", "year"]).reset_index(drop=True)
    sampled = (
        rows.groupby("study_site", group_keys=False)
            .apply(lambda g: g.sample(min(len(g), sample_per_site),
                                      random_state=seed))
    )
    sampled = sampled.sort_values(["study_site", "lake_id", "year"]).reset_index(drop=True)
    return [
        dict(lake_id=_norm_id(r["lake_id"]),
             year=int(r["year"]),
             study_site=str(r["study_site"]))
        for _, r in sampled.iterrows()
    ]

# ══════════════════════════════════════════════════════════════════════════════
#  THUMBNAIL LOADER
# ══════════════════════════════════════════════════════════════════════════════

def load_thumbnail(tif_path, lake_geom, lake_crs, thumb_w, thumb_h,
                   buffer_m=BOUNDARY_BUFFER_M):
    """
    Returns (thumb, rings, lake_valid).

    lake_valid is False only when NO pixels under the un-buffered lake polygon
    are valid (lake sits entirely under nodata).  Any other failure returns
    (None, [], True) so the caller shows a NOT FOUND placeholder instead of
    silently dropping the observation.
    """
    try:
        with rasterio.open(tif_path) as src:
            lake_proj = (gpd.GeoSeries([lake_geom], crs=lake_crs)
                         .to_crs(src.crs).geometry.iloc[0])

            # Validity check: un-buffered lake polygon only
            try:
                lake_only, _ = rio_mask(src, [lake_proj],
                                        crop=True, nodata=0, filled=True)
            except (ValueError, Exception):
                return None, [], False
            if lake_only.size == 0 or not np.any(lake_only > 0):
                return None, [], False

            # Buffered thumbnail for display
            out_img, out_transform = rio_mask(src, [lake_proj.buffer(buffer_m)],
                                              crop=True, nodata=0)
            if out_img.shape[0] < 3:
                return None, [], True
            clip_h = out_img.shape[1]
            clip_w = out_img.shape[2]
            rgb = np.dstack([_stretch(out_img[2]),
                             _stretch(out_img[1]),
                             _stretch(out_img[0])])

        bgr   = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        thumb = cv2.resize(bgr, (thumb_w, thumb_h), interpolation=cv2.INTER_AREA)

        inv_t   = ~out_transform
        scale_x = thumb_w / clip_w
        scale_y = thumb_h / clip_h

        polys = (lake_proj.geoms if lake_proj.geom_type == "MultiPolygon"
                 else [lake_proj])
        rings = []
        for poly in polys:
            for ring in [poly.exterior, *poly.interiors]:
                xs = np.array([c[0] for c in ring.coords])
                ys = np.array([c[1] for c in ring.coords])
                cols = inv_t.a * xs + inv_t.b * ys + inv_t.c
                rows = inv_t.d * xs + inv_t.e * ys + inv_t.f
                pts  = np.column_stack(
                    (np.round(cols * scale_x).astype(np.int32),
                     np.round(rows * scale_y).astype(np.int32))
                )
                rings.append(pts)

        return thumb, rings, True

    except Exception as exc:
        print(f"    Thumbnail error ({os.path.basename(tif_path)}): {exc}")
        return None, [], True


# ══════════════════════════════════════════════════════════════════════════════
#  LAKE-YEAR CONTAINER
# ══════════════════════════════════════════════════════════════════════════════

class LakeYearData:
    __slots__ = ("lake_id", "year", "study_site",
                 "obs", "thumbs", "outlines",
                 "lake_geom", "lake_crs", "img_lookup",
                 "fu", "divider_pos",
                 "thumb_w", "thumb_h",
                 "window_start", "show_all",
                 "cloudy_skipped")

    def __init__(self, **kw):
        self.window_start   = 0
        self.show_all       = False
        self.cloudy_skipped = 0
        for k, v in kw.items():
            setattr(self, k, v)

    @property
    def n(self):
        return len(self.obs)

    def _max_window_start(self):
        return max(0, self.n - WINDOW_SIZE)

    def _ensure_window_visible(self):
        if self.show_all or self.n <= WINDOW_SIZE:
            self.window_start = 0
            return
        rel = self.divider_pos - self.window_start
        if rel < 0:
            self.window_start = max(0, self.window_start - WINDOW_SHIFT)
        elif rel >= WINDOW_SIZE:
            self.window_start = min(self._max_window_start(),
                                    self.window_start + WINDOW_SHIFT)
        self.window_start = max(0, min(self._max_window_start(), self.window_start))

    def center_window_on_divider(self):
        if self.n <= WINDOW_SIZE:
            self.window_start = 0
            return
        target = max(self.divider_pos, 0) - (WINDOW_SIZE // 2 - 1)
        self.window_start = max(0, min(self._max_window_start(), target))

    def move_left(self):
        self.divider_pos = max(-1, self.divider_pos - 1)
        self._ensure_window_visible()

    def move_right(self):
        self.divider_pos = min(self.n - 1, self.divider_pos + 1)
        self._ensure_window_visible()

    def toggle_show_all(self):
        self.show_all = not self.show_all
        if not self.show_all:
            self.center_window_on_divider()


# ══════════════════════════════════════════════════════════════════════════════
#  PREPARE  (worker-safe)
# ══════════════════════════════════════════════════════════════════════════════

def prepare_lake_year(task, very_cloudy, df_results, img_lookups, shp_cache):
    lake_id    = task["lake_id"]
    year       = task["year"]
    study_site = task["study_site"]

    fu_rows = df_results[(df_results["lake_id"]    == lake_id) &
                         (df_results["year"]       == year)    &
                         (df_results["study_site"] == study_site)]
    if fu_rows.empty:
        print(f"    SKIP REASON: no matching row in df_results for {lake_id}/{year}/{study_site}")
        return None
    fu = fu_rows.iloc[0].to_dict()

    obs_list, cloudy_skipped = parse_obs_from_row(fu, very_cloudy)
    if not obs_list:
        obs_cols = [c for c in fu if c.startswith("Obs") and not any(
            c.startswith(p) for p in ("ObsK",)) and c[3:].split("Date")[0].split("PctIce")[0].split("PctWat")[0].isdigit()]
        print(f"    SKIP REASON: no valid obs for {lake_id}/{year}/{study_site}  "
              f"(cloudy_skipped={cloudy_skipped}, obs_cols_found={obs_cols[:6]})")
        return None

    obs_list.sort(key=lambda r: r["datetime"] if pd.notna(r["datetime"])
                  else pd.Timestamp.min)

    shp_path = SITE_SHAPEFILES.get(study_site)
    if not shp_path or not os.path.exists(shp_path):
        print(f"    SKIP REASON: shapefile not found: {shp_path}")
        return None
    if shp_path not in shp_cache:
        shp_cache[shp_path] = gpd.read_file(shp_path)
    gdf = shp_cache[shp_path]
    id_col = next((c for c in ("id", "ID", "lake_id", "LAKE_ID", "FID", "OBJECTID")
                   if c in gdf.columns), None)
    if id_col is None:
        print(f"    SKIP REASON: no recognised ID column in shapefile (cols: {list(gdf.columns)})")
        return None
    lake_rows = gdf[gdf[id_col].apply(_norm_id) == lake_id]
    if lake_rows.empty:
        print(f"    SKIP REASON: lake_id {lake_id!r} not found in shapefile column {id_col!r}")
        return None
    lake_geom = lake_rows.geometry.iloc[0]
    lake_crs  = gdf.crs

    img_lookup = img_lookups.get(study_site, {})
    for row in obs_list:
        tif_path = img_lookup.get(row["image_name"])
        row["tif_filename"] = os.path.basename(tif_path) if tif_path else row["image_name"]

    n           = len(obs_list)
    available_w = MAX_WINDOW_W - DIVIDER_W
    visible_n   = min(WINDOW_SIZE, n) if n > 0 else 1
    cell_w      = max(MIN_THUMB_W, available_w // visible_n)
    thumb_w     = cell_w
    thumb_h     = max(80, int(280 * cell_w / 280))
    cell_h      = thumb_h + LABEL_H
    total_h     = HEADER_H + cell_h
    if total_h > MAX_WINDOW_H:
        scale   = (MAX_WINDOW_H - HEADER_H - LABEL_H) / thumb_h
        thumb_h = max(80, int(thumb_h * scale))
        thumb_w = max(MIN_THUMB_W, int(thumb_w * scale))

    filtered_obs = []
    thumbs, outlines = [], []
    n_dropped = 0
    for row in obs_list:
        tif_path = img_lookup.get(row["image_name"])
        if tif_path:
            t, r, lake_valid = load_thumbnail(
                tif_path, lake_geom, lake_crs, thumb_w, thumb_h
            )
        else:
            t, r, lake_valid = None, [], True
        if not lake_valid:
            n_dropped += 1
            continue
        filtered_obs.append(row)
        thumbs.append(t)
        outlines.append(r)

    if n_dropped:
        print(f"    Dropped {n_dropped} obs with lake under nodata border.")

    if not filtered_obs:
        print(f"    SKIP REASON: all {len(obs_list)} obs dropped (lake under nodata) "
              f"for {lake_id}/{year}/{study_site}")
        return None

    obs_list = filtered_obs

    ref_date = fu.get("obs_before_date")
    if pd.isna(ref_date) if ref_date is not None else True:
        ref_date = fu.get("breakup_date")

    divider_pos = -1
    if ref_date is not None and pd.notna(ref_date):
        ref = pd.Timestamp(ref_date).normalize()
        for i, row in enumerate(obs_list):
            if pd.notna(row["datetime"]) and \
               pd.Timestamp(row["datetime"]).normalize() <= ref:
                divider_pos = i

    divider_pos = max(-1, min(divider_pos, len(obs_list) - 1))

    lyd = LakeYearData(
        lake_id=lake_id, year=year, study_site=study_site,
        obs=obs_list, thumbs=thumbs, outlines=outlines,
        lake_geom=lake_geom, lake_crs=lake_crs,
        img_lookup=img_lookup, fu=fu, divider_pos=divider_pos,
        thumb_w=thumb_w, thumb_h=thumb_h,
        cloudy_skipped=cloudy_skipped,
    )
    lyd.center_window_on_divider()
    return lyd

def preload_worker(task_q, result_q, very_cloudy, df_results, img_lookups, shp_cache):
    while True:
        task = task_q.get()
        if task is None:
            break
        key  = (_norm_id(task["lake_id"]), task["year"], task["study_site"])
        data = None
        try:
            data = prepare_lake_year(task, very_cloudy, df_results,
                                     img_lookups, shp_cache)
        except Exception as exc:
            print(f"  Pre-load error {key}: {exc}")
        result_q.put((key, data))
        task_q.task_done()


# ══════════════════════════════════════════════════════════════════════════════
#  ZOOM RELOAD
# ══════════════════════════════════════════════════════════════════════════════

def _reload_thumbs(lyd, buffer_m):
    new_thumbs, new_outlines = [], []
    for row in lyd.obs:
        tif_path = lyd.img_lookup.get(row["image_name"])
        if tif_path:
            t, r, _valid = load_thumbnail(tif_path, lyd.lake_geom, lyd.lake_crs,
                                          lyd.thumb_w, lyd.thumb_h, buffer_m)
        else:
            t, r = None, []
        new_thumbs.append(t)
        new_outlines.append(r)
    lyd.thumbs   = new_thumbs
    lyd.outlines = new_outlines


# ══════════════════════════════════════════════════════════════════════════════
#  RENDERING
# ══════════════════════════════════════════════════════════════════════════════

def _mid_date(lyd):
    dp = lyd.divider_pos
    n  = lyd.n
    b  = lyd.obs[dp]         if 0 <= dp < n     else None
    a  = lyd.obs[dp + 1]     if 0 <= dp + 1 < n else None
    if b is None and a is None:
        return None
    if b is None:
        return pd.Timestamp(a["datetime"])
    if a is None:
        return pd.Timestamp(b["datetime"])
    t0, t1 = pd.Timestamp(b["datetime"]), pd.Timestamp(a["datetime"])
    return t0 + (t1 - t0) / 2


def _flanking_fnames(lyd):
    dp = lyd.divider_pos
    n  = lyd.n
    bf = str(lyd.obs[dp]["tif_filename"])     if 0 <= dp < n     else ""
    af = str(lyd.obs[dp + 1]["tif_filename"]) if 0 <= dp + 1 < n else ""
    return bf, af


def _draw_divider(frame, x, y, w, h, lyd, man_s):
    cv2.rectangle(frame, (x, y), (x + w - 1, y + h - 1), C_DIVIDER, -1)
    tx = x + w // 2 - 7
    for j, ch in enumerate("BREAK"):
        _put_text(frame, ch, tx, y + 22 + j * 18, scale=0.40,
                  colour=C_WHITE, thickness=1)
    _put_text(frame, "UP", tx - 2, y + 22 + 5 * 18, scale=0.40, colour=C_WHITE)
    _put_text(frame, man_s[:6], tx - 8, y + h - 28, scale=0.35,
              colour=(220, 220, 255))
    _put_text(frame, man_s[7:] if len(man_s) > 6 else "", tx - 5, y + h - 14,
              scale=0.35, colour=(220, 220, 255))


def render_frame(lyd, current_idx, total, show_outline=True, zoom_tight=False):
    n  = lyd.n
    dp = lyd.divider_pos

    # ── Visible slice & per-thumb display size ───────────────────────────────
    if lyd.show_all or n <= WINDOW_SIZE:
        vis_start = 0
        vis_count = n
        if n > 0:
            avail  = MAX_WINDOW_W - DIVIDER_W
            target = max(4, avail // max(n, 1))
            disp_w = min(lyd.thumb_w, target)
        else:
            disp_w = lyd.thumb_w
        disp_h = (int(lyd.thumb_h * disp_w / lyd.thumb_w)
                  if lyd.thumb_w > 0 else lyd.thumb_h)
    else:
        vis_start = lyd.window_start
        vis_count = min(WINDOW_SIZE, n - vis_start)
        disp_w    = lyd.thumb_w
        disp_h    = lyd.thumb_h

    show_labels   = disp_w >= 60
    show_outlines = show_outline and disp_w >= 50

    cell_h   = (disp_h + LABEL_H) if show_labels else disp_h
    canvas_w = max(MAX_WINDOW_W // 4, vis_count * disp_w + DIVIDER_W)
    canvas_h = HEADER_H + cell_h + 4

    frame = np.full((canvas_h, canvas_w, 3), C_BG, dtype=np.uint8)

    # ── Header ───────────────────────────────────────────────────────────────
    frame[:HEADER_H, :] = C_HEADER
    orig_dt = lyd.fu.get("breakup_date")
    orig_s  = pd.Timestamp(orig_dt).strftime("%b %d %Y") if pd.notna(orig_dt) else "N/A"
    mid     = _mid_date(lyd)
    if mid is not None:
        man_s = mid.strftime("%b %d %Y")
    elif dp == -1:
        man_s = "before 1st obs"
    else:
        man_s = "?"

    if lyd.show_all or n <= WINDOW_SIZE:
        view_s = f"showing all {n}"
    else:
        last_visible = min(vis_start + WINDOW_SIZE, n)
        view_s = f"images {vis_start + 1}-{last_visible} of {n}"

    # Cloudy-skipped notice rendered in amber so it stands out
    cloudy_s = (f"   !! {lyd.cloudy_skipped} cloudy skipped !!"
                if lyd.cloudy_skipped > 0 else "")

    line1 = (f"  Lake {lyd.lake_id}   {lyd.year}   {lyd.study_site}     "
             f"Original break-up: {orig_s}     "
             f"Manual: {man_s}     "
             f"[{view_s}]     ({current_idx} / {total})"
             f"{cloudy_s}")
    zoom_s    = "ON (40m)" if zoom_tight   else "off"
    outline_s = "ON"       if show_outline else "off"
    showall_s = "ON"       if lyd.show_all else "off"
    line2 = (f"  Controls:  <- / A  left    -> / D  right    "
             f"] = show all [{showall_s}]    "
             f"Enter = save    X = no date    B = back    "
             f"O = outline [{outline_s}]    Z = zoom [{zoom_s}]    ESC = quit")
    _put_text(frame, line1, 6, 18, scale=0.50,
              colour=C_WARN if lyd.cloudy_skipped > 0 else C_WHITE)
    _put_text(frame, line2, 6, 40, scale=0.40, colour=C_DIM)

    # ── Body: thumbnails + divider ───────────────────────────────────────────
    x  = 0
    cy = HEADER_H

    need_resize   = (disp_w != lyd.thumb_w) or (disp_h != lyd.thumb_h)
    outline_scale = (disp_w / lyd.thumb_w, disp_h / lyd.thumb_h) \
                    if lyd.thumb_w > 0 else (1.0, 1.0)

    for i_local in range(vis_count):
        i_global = vis_start + i_local

        if i_global == dp + 1:
            _draw_divider(frame, x, cy, DIVIDER_W, cell_h, lyd, man_s)
            x += DIVIDER_W

        frame[cy:cy + cell_h, x:x + disp_w] = C_CELL

        thumb = lyd.thumbs[i_global]
        if thumb is not None:
            t = cv2.resize(thumb, (disp_w, disp_h),
                           interpolation=cv2.INTER_AREA) if need_resize else thumb

            if show_outlines and lyd.outlines[i_global]:
                t = t.copy()
                for pts in lyd.outlines[i_global]:
                    spts = (pts * np.array(outline_scale)).astype(np.int32) \
                           if need_resize else pts
                    cv2.polylines(t, [spts], isClosed=True,
                                  color=(0, 0, 0), thickness=3, lineType=cv2.LINE_AA)
                for pts in lyd.outlines[i_global]:
                    spts = (pts * np.array(outline_scale)).astype(np.int32) \
                           if need_resize else pts
                    cv2.polylines(t, [spts], isClosed=True,
                                  color=(0, 255, 255), thickness=1, lineType=cv2.LINE_AA)

            frame[cy:cy + disp_h, x:x + disp_w] = t
        elif show_labels:
            _put_text(frame, "NOT FOUND",
                      x + disp_w // 2 - 35, cy + disp_h // 2,
                      scale=0.45, colour=(100, 100, 160))

        if show_labels:
            row    = lyd.obs[i_global]
            dt_str = pd.Timestamp(row["datetime"]).strftime("%b %d %Y") \
                     if pd.notna(row["datetime"]) else "???"
            ly = cy + disp_h
            _put_text(frame, dt_str, x + 3, ly + 16, scale=0.44, colour=C_WHITE)

            wf = row.get("water_fraction")
            if wf is not None and not pd.isna(wf):
                col = C_WATER_HIGH if wf >= WATER_THRESHOLD else C_WATER_LOW
                _put_text(frame, f"{wf:.1f}% water",
                          x + 3, ly + 36, scale=0.44, colour=col)

        x += disp_w

    # Divider after the last visible image when dp is at that boundary
    if vis_count > 0 and dp == vis_start + vis_count - 1:
        _draw_divider(frame, x, cy, DIVIDER_W, cell_h, lyd, man_s)

    return frame


# ══════════════════════════════════════════════════════════════════════════════
#  OUTPUT CSV
# ══════════════════════════════════════════════════════════════════════════════

OUTPUT_COLS = ["study_site", "lake_id", "year",
               "original_breakup_date", "manual_breakup_date",
               "image_before", "image_after"]


def load_done(output_csv):
    if not os.path.exists(output_csv):
        return set(), []
    df   = pd.read_csv(output_csv)
    done = {(_norm_id(str(r["lake_id"])), int(r["year"]), str(r["study_site"]))
            for _, r in df.iterrows()}
    return done, df.to_dict("records")


def save_row(output_csv, record, existing_rows):
    existing_rows.append(record)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
        writer.writeheader()
        writer.writerows(existing_rows)


# ══════════════════════════════════════════════════════════════════════════════
#  KEY CODES
# ══════════════════════════════════════════════════════════════════════════════

_KEY_LEFT    = {2424832, 65361, 63234, ord('a'), ord('A')}
_KEY_RIGHT   = {2555904, 65363, 63235, ord('d'), ord('D')}
_KEY_ENTER   = {13, 10}
_KEY_ESC     = {27}
_KEY_X       = {ord('x'), ord('X')}
_KEY_B       = {ord('b'), ord('B')}
_KEY_O       = {ord('o'), ord('O')}
_KEY_Z       = {ord('z'), ord('Z')}
_KEY_BRACKET = {ord(']')}


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run():
    print("=" * 60)

    print("\nChecking configured paths ...")
    ok = True
    for ds, p in SITE_SHAPEFILES.items():
        print(f"  [{'OK' if os.path.exists(p) else 'MISSING'}]  {ds} shapefile: {p}")
        ok &= os.path.exists(p)
    for ds, p in SITE_FOLDERS.items():
        print(f"  [{'OK' if os.path.isdir(p) else 'MISSING'}]  {ds} imagery folder: {p}")
        ok &= os.path.isdir(p)
    for ds, p in BREAKUP_RESULTS_CSVS.items():
        print(f"  [{'OK' if os.path.exists(p) else 'MISSING'}]  {ds} results CSV: {p}")
        ok &= os.path.exists(p)
    if not ok:
        print("\n  One or more required paths are missing.")
        print("  Update the paths in USER CONFIG.")
        return

    print("\nLoading cloud filter ...")
    very_cloudy = load_very_cloudy()

    print("\nLoading break-up results ...")
    df_results = load_results(BREAKUP_RESULTS_CSVS)

    print("\nBuilding lake-year list ...")
    all_lys = build_lake_year_list(df_results)
    if not all_lys:
        print("ERROR: No lake-years found in results CSV.")
        return

    done_set, existing_rows = load_done(OUTPUT_CSV)
    todo = [t for t in all_lys
            if (_norm_id(t["lake_id"]), t["year"], t["study_site"]) not in done_set]

    print(f"  Total: {len(all_lys)}   Already done: {len(done_set)}   "
          f"To review: {len(todo)}")
    if not todo:
        print("All lake-years already reviewed!")
        return

    print("\nIndexing TIF files ...")
    img_lookups = {}
    for ds in {t["study_site"] for t in todo}:
        folder = SITE_FOLDERS.get(ds)
        if folder and os.path.isdir(folder):
            img_lookups[ds] = build_image_lookup(folder)
            print(f"  {ds}: ~{len(img_lookups[ds])//2:,} TIFs indexed")
        else:
            print(f"  WARNING: imagery folder not found for {ds}: {folder}")

    N_WORKERS = 4
    shp_cache = {}
    task_q    = queue.Queue()
    result_q  = queue.Queue()

    for _w in range(N_WORKERS):
        t = threading.Thread(
            target=preload_worker,
            args=(task_q, result_q, very_cloudy, df_results, img_lookups, shp_cache),
            daemon=True,
            name=f"PreloadWorker-{_w}",
        )
        t.start()

    n_enqueued = 0
    for i in range(min(PRELOAD_AHEAD, len(todo))):
        task_q.put(todo[i])
        n_enqueued += 1

    preload_cache = {}

    def _drain_results():
        while True:
            try:
                key, data = result_q.get_nowait()
                preload_cache[key] = data
            except queue.Empty:
                break

    def _get_lyd(idx):
        t   = todo[idx]
        key = (_norm_id(t["lake_id"]), t["year"], t["study_site"])
        while key not in preload_cache:
            _drain_results()
            if key not in preload_cache:
                try:
                    k2, data = result_q.get(timeout=0.15)
                    preload_cache[k2] = data
                except queue.Empty:
                    pass
        return preload_cache[key]

    WIN = "Break-up Manual Review"
    cv2.namedWindow(WIN, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WIN, min(MAX_WINDOW_W, 1600), MAX_WINDOW_H)

    total = len(todo)
    idx   = 0
    shown = 0

    print("\n" + "=" * 60)
    print("OpenCV window opened. <- -> move divider, ] = show all overview, "
          "Enter confirms, X = no date, ESC quits.")
    print("=" * 60)

    while idx < total:
        t = todo[idx]
        next_to_enqueue = idx + PRELOAD_AHEAD
        while n_enqueued <= next_to_enqueue and n_enqueued < total:
            task_q.put(todo[n_enqueued])
            n_enqueued += 1

        print(f"  Loading [{idx+1}/{total}]: "
              f"Lake {t['lake_id']}  {t['year']}  {t['study_site']} ...",
              end="  ", flush=True)
        lyd = _get_lyd(idx)
        if lyd is None:
            print("SKIPPED (no data)")
            idx += 1
            continue
        cloudy_note = (f"  [{lyd.cloudy_skipped} cloudy obs skipped]"
                       if lyd.cloudy_skipped > 0 else "")
        print(f"{lyd.n} observations ready.{cloudy_note}")

        show_outline = True
        zoom_tight   = False
        win_w, win_h = min(MAX_WINDOW_W, 1600), MAX_WINDOW_H

        while True:
            frame = render_frame(lyd, idx + 1, total,
                                 show_outline=show_outline, zoom_tight=zoom_tight)

            rect = cv2.getWindowImageRect(WIN)
            if rect[2] > 400 and rect[3] > 400:
                win_w, win_h = rect[2], rect[3]

            display = letterbox_frame(frame, win_w, win_h)
            cv2.imshow(WIN, display)
            key = cv2.waitKeyEx(30)

            if key == -1:
                continue

            if key in _KEY_LEFT:
                lyd.move_left()

            elif key in _KEY_RIGHT:
                lyd.move_right()

            elif key in _KEY_BRACKET:
                lyd.toggle_show_all()

            elif key in _KEY_ENTER:
                mid    = _mid_date(lyd)
                man_s  = mid.strftime("%Y-%m-%d") if mid is not None else ""
                bf, af = _flanking_fnames(lyd)
                orig   = lyd.fu.get("breakup_date")
                orig_s = pd.Timestamp(orig).strftime("%Y-%m-%d") if pd.notna(orig) else ""
                record = {
                    "study_site":            lyd.study_site,
                    "lake_id":               lyd.lake_id,
                    "year":                  lyd.year,
                    "original_breakup_date": orig_s,
                    "manual_breakup_date":   man_s,
                    "image_before":          bf,
                    "image_after":           af,
                }
                save_row(OUTPUT_CSV, record, existing_rows)
                print(f"  + Saved  {lyd.lake_id}/{lyd.year}  ->  {man_s}")
                shown += 1
                idx   += 1
                break

            elif key in _KEY_X:
                orig   = lyd.fu.get("breakup_date")
                orig_s = pd.Timestamp(orig).strftime("%Y-%m-%d") if pd.notna(orig) else ""
                record = {
                    "study_site":            lyd.study_site,
                    "lake_id":               lyd.lake_id,
                    "year":                  lyd.year,
                    "original_breakup_date": orig_s,
                    "manual_breakup_date":   "NO MANUAL DATE",
                    "image_before":          "",
                    "image_after":           "",
                }
                save_row(OUTPUT_CSV, record, existing_rows)
                print(f"  x Saved  {lyd.lake_id}/{lyd.year}  ->  NO MANUAL DATE")
                shown += 1
                idx   += 1
                break

            elif key in _KEY_B:
                if shown == 0:
                    print("  Nothing to undo this session.")
                else:
                    existing_rows.pop()
                    if existing_rows:
                        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
                            writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
                            writer.writeheader()
                            writer.writerows(existing_rows)
                    else:
                        try:
                            os.remove(OUTPUT_CSV)
                        except OSError:
                            pass
                    shown -= 1
                    idx   -= 1
                    prev_t = todo[idx]
                    print(f"  Undone -- back to Lake {prev_t['lake_id']}  "
                          f"{prev_t['year']}  {prev_t['study_site']}")
                    break

            elif key in _KEY_O:
                show_outline = not show_outline

            elif key in _KEY_Z:
                zoom_tight = not zoom_tight
                buf = ZOOM_TIGHT_BUFFER if zoom_tight else BOUNDARY_BUFFER_M
                print(f"  Zoom {'tight (40 m)' if zoom_tight else 'default (500 m)'} "
                      f"-- reloading thumbnails ...", end="  ", flush=True)
                _reload_thumbs(lyd, buf)
                print("done.")

            elif key in _KEY_ESC:
                print(f"\n  ESC -- quitting after {shown} reviewed.")
                cv2.destroyAllWindows()
                for _ in range(N_WORKERS):
                    task_q.put(None)
                return

    cv2.destroyAllWindows()
    for _ in range(N_WORKERS):
        task_q.put(None)
    print(f"\n{'='*60}")
    print(f"  Completed!  {shown} lake-years reviewed.")
    print(f"  Results saved to: {OUTPUT_CSV}")
    print("=" * 60)


if __name__ == "__main__":
    run()

NameError: name 'SITES' is not defined